# CS 5326 PA3: full evaluation pipeline

Runs every configuration on the 50-problem benchmark (`data/gsm8k_test_50.jsonl`) and writes the artifacts in `outputs/`:
`per_problem.jsonl`, `results_summary.json`, `aggregation_ablation.json`, `sc_scaling_curve.json`,
plus `bonus_logprob.json`, `report_stats.json` and `env.json`.

Per problem, in a fixed order: seed `5326 + problem_index`; greedy; 5 sampled paths (the shared pool, SC@5);
10 more samples (compute-matched SC@15); every step of the 5 pool paths judged in one batched pass, replayed
into `run_prm_best_of_n`; then `select_best_path` for min, product, step_penalty (lambda 0.0, 0.5, 2.0) and the
optional `product_logprob` bonus. One line is appended to `outputs/per_problem.jsonl` after each problem, so a
rerun skips finished problems.

**Kaggle:** Accelerator GPU T4 x2, Internet on, then Save Version, Save & Run All. The outputs are zipped to
`/kaggle/working/pa3_outputs.zip`. To resume after a disconnect, attach the previous version's output as input:
its `per_problem.jsonl` is copied in and finished problems are skipped.

In [ ]:
# [cell: config]
N_PROBLEMS = 50
OUT_DIR = "outputs"            # relative to the repository root, as in the manual
JUDGE_BATCH_SIZE = 16
POOL_SIZE = 5                  # shared candidates: SC@5 and PRM Best-of-5
SC_TARGET = 15                 # compute-matched self-consistency
TEMPERATURE = 0.7
SELECTIONS = [("min", None), ("product", None), ("step_penalty", 0.0), ("step_penalty", 0.5), ("step_penalty", 2.0)]
BOOTSTRAP_RESAMPLES = 1000
BOOTSTRAP_SEED = 5326
ARCHIVE_NAME = "pa3_outputs"

In [ ]:
# [cell: setup]
import glob
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/mabdullahi7780/cs5326-pa3-starter.git"
BRANCH = "main"


def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check)


if os.path.isdir("src") and os.path.isfile("data/gsm8k_test_50.jsonl"):
    REPO_DIR = os.getcwd()                    # already at the repository root
else:
    REPO_DIR = "/tmp/cs5326-pa3-starter"      # Kaggle: clone outside /kaggle/working
    if not os.path.isdir(REPO_DIR):
        run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nsubject %s"], cwd=REPO_DIR, check=False)

if os.path.isdir("/kaggle"):
    # Manual section 3.4: Kaggle already has torch, transformers and accelerate; only bitsandbytes is upgraded.
    run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"])

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

# Resume: a per_problem.jsonl from an attached earlier version is copied in once.
PER_PROBLEM = f"{OUT_DIR}/per_problem.jsonl"
previous = sorted(glob.glob("/kaggle/input/**/per_problem.jsonl", recursive=True))
if previous and not os.path.exists(PER_PROBLEM):
    shutil.copy(previous[0], PER_PROBLEM)
    print("resuming from", previous[0])

In [ ]:
# [cell: env]
import importlib.metadata as metadata
import platform

import torch
import transformers

env = {
    "commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "accelerate": metadata.version("accelerate"),
    "bitsandbytes": metadata.version("bitsandbytes"),
    "cuda": torch.version.cuda,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
print(json.dumps(env, indent=2))
assert torch.cuda.device_count() >= 1, "No GPU: set the accelerator to GPU T4 x2"
with open(f"{OUT_DIR}/env.json", "w") as handle:
    json.dump(env, handle, indent=2)

In [ ]:
# [cell: load]
import time

from src.utils import judge_prompts_batched, load_models, make_hf_generate_fn, verdict_token_pairs

load_start = time.perf_counter()
gen_model, gen_tokenizer, judge_model, judge_tokenizer = load_models()
LOAD_SECONDS = time.perf_counter() - load_start
print(f"models loaded in {LOAD_SECONDS:.0f}s")
print("generator:", gen_model.device, gen_model.dtype)
print("judge:", judge_model.device, "4-bit:", getattr(judge_model, "is_loaded_in_4bit", None))
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} allocated {torch.cuda.memory_allocated(i) / 2**30:.2f} GiB")

# Bonus: the verdict words must be single tokens (expected ((43291, 32269), (10044, 46859))).
VERDICT_PAIRS = verdict_token_pairs(judge_tokenizer)
print("verdict token ids (valid, invalid):", VERDICT_PAIRS)

GENERATE_FN = make_hf_generate_fn(gen_model, gen_tokenizer)


def JUDGE_BATCH(prompts):
    """One batched greedy pass over judge prompts, with the bonus verdict probabilities."""
    return judge_prompts_batched(judge_model, judge_tokenizer, prompts,
                                 batch_size=JUDGE_BATCH_SIZE, return_verdict_probs=True)

In [ ]:
# [cell: run]
import time
import warnings

from src.generation import compute_majority_vote, run_greedy, run_self_consistency
from src.prm_judge import judge_prompts_for_path
from src.search import (aggregate_path_score, run_compute_matched_self_consistency,
                        run_prm_best_of_n, select_best_path)
from src.utils import GLOBAL_SEED, make_replay_judge_fn, set_seed_for_problem

with open("data/gsm8k_test_50.jsonl") as handle:
    problems = [json.loads(line) for line in handle if line.strip()][:N_PROBLEMS]
assert len(problems) == N_PROBLEMS


def selection_key(strategy, lam):
    return strategy if lam is None else f"{strategy}_{lam}"


def is_correct(answer, truth):
    return answer is not None and answer == truth


def clamp_verdict(score):
    """Bonus fallback when no verdict token was generated: the clamped discrete verdict."""
    return min(max((score + 1.0) / 2.0, 0.01), 0.99)


def run_problem(index, problem):
    question, truth = problem["question"], problem["ground_truth_answer"]
    set_seed_for_problem(index)

    # 1. Generation, all of it before any judging.
    greedy = run_greedy(question=question, generate_fn=GENERATE_FN)
    sc5 = run_self_consistency(question=question, n=POOL_SIZE, temperature=TEMPERATURE, generate_fn=GENERATE_FN)
    sc15 = run_compute_matched_self_consistency(question=question, target_calls=SC_TARGET, temperature=TEMPERATURE,
                                                existing_candidates=sc5["candidates"], generate_fn=GENERATE_FN)
    pool = sc5["candidates"]

    # 2. Judging: one batched pass over every step of the pool, replayed into run_prm_best_of_n.
    prompts = [p for candidate in pool for p in judge_prompts_for_path(question, candidate["steps"])]
    start = time.perf_counter()
    judged = JUDGE_BATCH(prompts)
    batch_seconds = time.perf_counter() - start
    replay = make_replay_judge_fn(prompts, judged)
    prm = run_prm_best_of_n(question=question, n=POOL_SIZE, aggregation_strategy="min",
                            candidates=pool, judge_fn=replay)
    assert replay.remaining() == 0 and prm["generation_calls"] == 0
    assert prm["judge_calls"] == len(prompts) == sum(len(c["steps"]) for c in pool)

    # 3. Per-candidate judge details and the bonus probabilities.
    judged_pool, offset, fallbacks = [], 0, 0
    for k, candidate in enumerate(prm["candidates"]):
        outputs = judged[offset:offset + len(candidate["steps"])]
        offset += len(candidate["steps"])
        probs = []
        for output, score in zip(outputs, candidate["step_scores"]):
            if output.get("verdict_prob") is None:
                fallbacks += 1
                probs.append(clamp_verdict(score))
            else:
                probs.append(output["verdict_prob"])
        judged_pool.append(dict(candidate, pool_index=k,
                                judge_texts=[o["text"] for o in outputs],
                                judge_tokens=[o["num_tokens"] for o in outputs],
                                raw_verdict_probs=[o.get("verdict_prob") for o in outputs],
                                verdict_probs=probs))

    # 4. Selection: every strategy and lambda on the same judged pool, each timed on its own.
    methods = {}
    for strategy, lam in SELECTIONS:
        start = time.perf_counter()
        chosen = select_best_path(judged_pool, strategy, 2.0 if lam is None else lam)
        seconds = time.perf_counter() - start
        methods["prm_" + selection_key(strategy, lam)] = {
            "selected_index": chosen["pool_index"], "answer": chosen.get("answer"),
            "path_score": chosen["path_score"], "num_tied": chosen["num_tied"],
            "correct": is_correct(chosen.get("answer"), truth), "select_seconds": seconds}
    assert methods["prm_min"]["answer"] == prm["answer"]
    bonus_pool = [dict(c, step_scores=c["verdict_probs"]) for c in judged_pool]
    start = time.perf_counter()
    chosen = select_best_path(bonus_pool, "product_logprob")
    methods["prm_product_logprob"] = {
        "selected_index": chosen["pool_index"], "answer": chosen.get("answer"),
        "path_score": chosen["path_score"], "num_tied": chosen["num_tied"],
        "correct": is_correct(chosen.get("answer"), truth), "select_seconds": time.perf_counter() - start}
    methods["greedy"] = {"answer": greedy["answer"], "correct": is_correct(greedy["answer"], truth)}
    methods["sc5"] = {"answer": sc5["answer"], "correct": is_correct(sc5["answer"], truth)}
    methods["sc15"] = {"answer": sc15["answer"], "correct": is_correct(sc15["answer"], truth)}

    # 5. The record: all 15 candidates; judge details and path scores for the pool (0 to 4).
    candidates = []
    for k, candidate in enumerate(sc15["candidates"]):
        entry = {"index": k, "completion": candidate["completion"], "steps": candidate["steps"],
                 "answer": candidate["answer"], "num_tokens": candidate["num_tokens"],
                 "correct": is_correct(candidate["answer"], truth)}
        if k < POOL_SIZE:
            j = judged_pool[k]
            entry.update({
                "step_scores": j["step_scores"], "rationales": j["rationales"], "judge_texts": j["judge_texts"],
                "judge_tokens": j["judge_tokens"], "raw_verdict_probs": j["raw_verdict_probs"],
                "verdict_probs": j["verdict_probs"],
                "path_scores": {selection_key(s, l): aggregate_path_score(j["step_scores"], s, 2.0 if l is None else l)
                                for s, l in SELECTIONS},
                "bonus_path_score": aggregate_path_score(j["verdict_probs"], "product_logprob")})
        candidates.append(entry)
    answers = [c["answer"] for c in candidates]
    curve_answers = [compute_majority_vote(answers[:n]) for n in range(1, SC_TARGET + 1)]

    return {
        "index": index, "id": problem["id"], "seed": GLOBAL_SEED + index, "question": question,
        "ground_truth": truth,
        "greedy": {"completion": greedy["completion"], "steps": greedy["steps"], "answer": greedy["answer"],
                   "num_tokens": greedy["generated_tokens"], "correct": methods["greedy"]["correct"]},
        "candidates": candidates,
        "methods": methods,
        "costs": {
            "greedy": {"gen_calls": greedy["generation_calls"], "gen_tokens": greedy["generated_tokens"],
                       "seconds": greedy["time_seconds"]},
            "sc5": {"gen_calls": sc5["generation_calls"], "gen_tokens": sc5["generated_tokens"],
                    "seconds": sc5["time_seconds"]},
            "sc15_extra": {"gen_calls": sc15["generation_calls"], "gen_tokens": sc15["generated_tokens"],
                           "seconds": sc15["time_seconds"]},
            "judge": {"calls": prm["judge_calls"], "tokens": prm["judge_tokens"],
                      "batch_seconds": batch_seconds, "replay_seconds": prm["time_seconds"]},
        },
        "curve": {"answers": curve_answers, "correct": [is_correct(a, truth) for a in curve_answers]},
        "bonus": {"fallback_steps": fallbacks, "steps": len(prompts)},
    }


def fail_fast(record):
    """Stop within minutes, not after an hour, if the first problem of this session looks wrong."""
    problems_found = []
    if len(record["candidates"]) != SC_TARGET:
        problems_found.append(f"{len(record['candidates'])} candidates, expected {SC_TARGET}")
    pool = record["candidates"][:POOL_SIZE]
    for c in pool:
        if not (len(c["step_scores"]) == len(c["steps"]) == len(c["judge_texts"]) == len(c["verdict_probs"])):
            problems_found.append(f"candidate {c['index']}: steps and judge outputs differ in length")
        if any(not 0.0 <= p <= 1.0 for p in c["verdict_probs"]):
            problems_found.append(f"candidate {c['index']}: a probability outside [0, 1]")
    pairs = [(p, s) for c in pool for p, s in zip(c["raw_verdict_probs"], c["step_scores"]) if p is not None]
    agree = sum((p > 0.5) == (s > 0) for p, s in pairs)
    steps = record["bonus"]["steps"]
    print(f"fail-fast: {steps} judged steps, {record['bonus']['fallback_steps']} bonus fallbacks, "
          f"probability agrees with verdict on {agree}/{len(pairs)}")
    if steps and record["bonus"]["fallback_steps"] * 2 > steps:
        problems_found.append("more than half the judge outputs have no verdict token")
    if pairs and agree < 0.9 * len(pairs):
        problems_found.append("verdict probabilities disagree with the verdicts on over 10% of steps")
    if problems_found:
        raise RuntimeError("fail-fast check failed: " + "; ".join(problems_found))


done = set()
if os.path.exists(PER_PROBLEM):
    with open(PER_PROBLEM) as handle:
        done = {json.loads(line)["id"] for line in handle if line.strip()}
print(f"{len(done)} problems already done")

checked = False
run_start = time.perf_counter()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    for index, problem in enumerate(problems):
        if problem["id"] in done:
            continue
        start = time.perf_counter()
        record = run_problem(index, problem)
        record["wall_seconds"] = time.perf_counter() - start
        if not checked:
            fail_fast(record)
            checked = True
        with open(PER_PROBLEM, "a") as handle:
            handle.write(json.dumps(record) + "\n")
        m = record["methods"]
        print(f"{index:2d} {problem['id']} gt={record['ground_truth']} greedy={m['greedy']['answer']} "
              f"sc5={m['sc5']['answer']} sc15={m['sc15']['answer']} prm_min={m['prm_min']['answer']} "
              f"judge_calls={record['costs']['judge']['calls']} {record['wall_seconds']:.1f}s", flush=True)
RUN_SECONDS = time.perf_counter() - run_start
RUN_WARNINGS = sorted({f"{w.category.__name__}: {str(w.message)[:300]}" for w in caught})
print(f"loop wall time {RUN_SECONDS:.0f}s; {len(RUN_WARNINGS)} distinct warnings")
for message in RUN_WARNINGS:
    print(" ", message)

In [ ]:
# [cell: summary]
import random

from src.prm_judge import _VERDICT_LINE_RE
from src.utils import save_aggregation_ablation, save_results_summary, save_sc_scaling_curve

with open(PER_PROBLEM) as handle:
    rows = sorted((json.loads(line) for line in handle if line.strip()), key=lambda r: r["index"])
ids = [r["id"] for r in rows]
assert len(rows) == N_PROBLEMS and len(set(ids)) == N_PROBLEMS and ids == [p["id"] for p in problems]
n = len(rows)
STRATEGY_KEYS = [("prm_" + selection_key(s, l), s, l) for s, l in SELECTIONS]


def total(path):
    out = 0
    for r in rows:
        value = r
        for key in path:
            value = value[key]
        out += value
    return out


def accuracy(method):
    return 100.0 * sum(r["methods"][method]["correct"] for r in rows) / n


def tie_rate(method):
    return 100.0 * sum(r["methods"][method]["num_tied"] > 1 for r in rows) / n


greedy_calls, greedy_tokens = total(["costs", "greedy", "gen_calls"]), total(["costs", "greedy", "gen_tokens"])
pool_calls, pool_tokens = total(["costs", "sc5", "gen_calls"]), total(["costs", "sc5", "gen_tokens"])
extra_calls, extra_tokens = total(["costs", "sc15_extra", "gen_calls"]), total(["costs", "sc15_extra", "gen_tokens"])
judge_calls, judge_tokens = total(["costs", "judge", "calls"]), total(["costs", "judge", "tokens"])
greedy_seconds, pool_seconds = total(["costs", "greedy", "seconds"]), total(["costs", "sc5", "seconds"])
extra_seconds = total(["costs", "sc15_extra", "seconds"])
judging_seconds = total(["costs", "judge", "batch_seconds"]) + total(["costs", "judge", "replay_seconds"])


def row(method, accuracy_pct, gen_calls, j_calls, gen_tok, j_tok, seconds):
    return {"Method": method, "Accuracy (%)": accuracy_pct, "Generation Calls": gen_calls, "Judge Calls": j_calls,
            "Total LLM Calls": gen_calls + j_calls, "Avg Gen Tokens / Problem": gen_tok / n,
            "Avg Judge Tokens / Problem": j_tok / n, "Total Time (s)": seconds}


def prm_row(method, key):
    select_seconds = sum(r["methods"][key]["select_seconds"] for r in rows)
    return row(method, accuracy(key), pool_calls, judge_calls, pool_tokens, judge_tokens,
               pool_seconds + judging_seconds + select_seconds)


results_summary = [
    row("Greedy (N=1)", accuracy("greedy"), greedy_calls, 0, greedy_tokens, 0, greedy_seconds),
    row("Self-Consistency (N=5)", accuracy("sc5"), pool_calls, 0, pool_tokens, 0, pool_seconds),
    row("Compute-Matched SC (N=15)", accuracy("sc15"), pool_calls + extra_calls, 0,
        pool_tokens + extra_tokens, 0, pool_seconds + extra_seconds),
    prm_row("PRM Best-of-5 (min)", "prm_min"),
    prm_row("PRM Best-of-5 (product)", "prm_product"),
    prm_row("PRM Best-of-5 (step_penalty, lambda=2.0)", "prm_step_penalty_2.0"),
]
aggregation_ablation = [{"Strategy": s, "Lambda": l, "Accuracy (%)": accuracy(key), "Tie Rate (%)": tie_rate(key)}
                        for key, s, l in STRATEGY_KEYS]
sc_scaling_curve = [{"N": k, "Accuracy (%)": 100.0 * sum(r["curve"]["correct"][k - 1] for r in rows) / n,
                     "Avg Gen Tokens / Problem": sum(sum(c["num_tokens"] for c in r["candidates"][:k]) for r in rows) / n}
                    for k in range(1, SC_TARGET + 1)]

# Consistency checks (manual 5.3 and 5.4).
assert all(len(r["candidates"]) == SC_TARGET for r in rows)
assert greedy_calls == n and pool_calls == POOL_SIZE * n and pool_calls + extra_calls == SC_TARGET * n
assert judge_calls == sum(len(c["steps"]) for r in rows for c in r["candidates"][:POOL_SIZE])
assert all(r["curve"]["answers"][POOL_SIZE - 1] == r["methods"]["sc5"]["answer"] for r in rows)
assert all(r["curve"]["answers"][SC_TARGET - 1] == r["methods"]["sc15"]["answer"] for r in rows)
assert sc_scaling_curve[POOL_SIZE - 1]["Accuracy (%)"] == results_summary[1]["Accuracy (%)"]
assert sc_scaling_curve[SC_TARGET - 1]["Accuracy (%)"] == results_summary[2]["Accuracy (%)"]
assert abs(sc_scaling_curve[POOL_SIZE - 1]["Avg Gen Tokens / Problem"] - results_summary[1]["Avg Gen Tokens / Problem"]) < 1e-9
assert abs(sc_scaling_curve[SC_TARGET - 1]["Avg Gen Tokens / Problem"] - results_summary[2]["Avg Gen Tokens / Problem"]) < 1e-9
for i in (3, 4, 5):
    assert [results_summary[i][k] for k in ("Generation Calls", "Judge Calls", "Avg Gen Tokens / Problem",
                                             "Avg Judge Tokens / Problem")] == \
           [results_summary[3][k] for k in ("Generation Calls", "Judge Calls", "Avg Gen Tokens / Problem",
                                             "Avg Judge Tokens / Problem")]
assert aggregation_ablation[0]["Accuracy (%)"] == results_summary[3]["Accuracy (%)"]
assert aggregation_ablation[1]["Accuracy (%)"] == results_summary[4]["Accuracy (%)"]
assert aggregation_ablation[4]["Accuracy (%)"] == results_summary[5]["Accuracy (%)"]

save_results_summary(results_summary, f"{OUT_DIR}/results_summary.json")
save_aggregation_ablation(aggregation_ablation, f"{OUT_DIR}/aggregation_ablation.json")
save_sc_scaling_curve(sc_scaling_curve, f"{OUT_DIR}/sc_scaling_curve.json")

# Bonus: product_logprob over first-token verdict probabilities.
pool_steps = [(p, s) for r in rows for c in r["candidates"][:POOL_SIZE]
              for p, s in zip(c["raw_verdict_probs"], c["step_scores"])]
with_prob = [(p, s) for p, s in pool_steps if p is not None]
bonus = {
    "Strategy": "product_logprob",
    "Approach": "first-token logits: ' VALID' and ' INVALID' are single tokens, so this equals the full-sequence comparison",
    "Verdict Token Ids": [list(pair) for pair in VERDICT_PAIRS],
    "Accuracy (%)": accuracy("prm_product_logprob"),
    "Tie Rate (%)": tie_rate("prm_product_logprob"),
    "Fallback Steps": sum(r["bonus"]["fallback_steps"] for r in rows),
    "Judged Steps": len(pool_steps),
    "Probability Agrees With Verdict (%)": 100.0 * sum((p > 0.5) == (s > 0) for p, s in with_prob) / max(1, len(with_prob)),
    "Versus min": {
        "bonus_right_min_wrong": sum(r["methods"]["prm_product_logprob"]["correct"] and not r["methods"]["prm_min"]["correct"] for r in rows),
        "min_right_bonus_wrong": sum(r["methods"]["prm_min"]["correct"] and not r["methods"]["prm_product_logprob"]["correct"] for r in rows),
        "same_answer": sum(r["methods"]["prm_product_logprob"]["answer"] == r["methods"]["prm_min"]["answer"] for r in rows),
    },
}
with open(f"{OUT_DIR}/bonus_logprob.json", "w") as handle:
    json.dump(bonus, handle, indent=2)

# Report statistics (manual 7.1 and 7.4): paired bootstrap, win/loss, the 10x greedy budget.
methods = ["greedy", "sc5", "sc15"] + [key for key, _, _ in STRATEGY_KEYS] + ["prm_product_logprob"]
correct = {m: [1.0 if r["methods"][m]["correct"] else 0.0 for r in rows] for m in methods}
# 1,000 resamples of the problems with seed 5326; the same resamples for every method, so differences are paired.
bootstrap_rng = random.Random(BOOTSTRAP_SEED)
resamples = [[bootstrap_rng.randrange(n) for _ in range(n)] for _ in range(BOOTSTRAP_RESAMPLES)]


def percentile(sorted_values, q):
    """Linear interpolation between closest ranks (numpy's default method)."""
    position = (len(sorted_values) - 1) * q / 100.0
    low = int(position)
    high = min(low + 1, len(sorted_values) - 1)
    return sorted_values[low] + (sorted_values[high] - sorted_values[low]) * (position - low)


def ci(values):
    stats = sorted(100.0 * sum(values[i] for i in sample) / n for sample in resamples)
    return [percentile(stats, 2.5), percentile(stats, 97.5)]


main_prm = ["prm_min", "prm_product", "prm_step_penalty_2.0"]
paired, differences = {}, {}
for prm_key in main_prm + ["prm_product_logprob"]:
    for sc_key in ("sc5", "sc15"):
        a, b = correct[prm_key], correct[sc_key]
        paired[f"{prm_key} vs {sc_key}"] = {"prm_won": sum(x == 1 and y == 0 for x, y in zip(a, b)),
                                            "sc_won": sum(x == 0 and y == 1 for x, y in zip(a, b)),
                                            "both_right": sum(x == 1 and y == 1 for x, y in zip(a, b)),
                                            "both_wrong": sum(x == 0 and y == 0 for x, y in zip(a, b))}
        diff = [x - y for x, y in zip(a, b)]
        differences[f"{prm_key} - {sc_key}"] = {"difference": 100.0 * sum(diff) / n, "ci95": ci(diff)}

budget = 10 * greedy_tokens
spending = {r_["Method"]: r_["Avg Gen Tokens / Problem"] * n + r_["Avg Judge Tokens / Problem"] * n
            for r_ in results_summary}
curve_tokens = {c["N"]: c["Avg Gen Tokens / Problem"] * n for c in sc_scaling_curve}
first_lines = [t.strip().splitlines()[0] if t.strip() else "" for r in rows for c in r["candidates"][:POOL_SIZE]
               for t in c["judge_texts"]]
all_paths = [r["greedy"] for r in rows] + [c for r in rows for c in r["candidates"]]
report_stats = {
    "n_problems": n,
    "accuracy_ci95": {m: {"accuracy": 100.0 * sum(correct[m]) / n, "ci95": ci(correct[m])} for m in methods},
    "bootstrap": {"resamples": BOOTSTRAP_RESAMPLES, "seed": BOOTSTRAP_SEED, "paired": True,
                  "rng": "random.Random(5326)", "percentile": "linear interpolation"},
    "differences": differences,
    "paired_win_loss": paired,
    "k_bar": judge_calls / pool_calls,
    "judge_tokens_per_call": judge_tokens / max(1, judge_calls),
    "budget_10x_greedy_tokens": budget,
    "spending_total_output_tokens": spending,
    "within_budget": {method: tokens <= budget for method, tokens in spending.items()},
    "largest_sc_n_within_budget": max([k for k, tokens in curve_tokens.items() if tokens <= budget], default=0),
    "answer_parsed_pct": 100.0 * sum(p["answer"] is not None for p in all_paths) / len(all_paths),
    "zero_step_paths": sum(len(p["steps"]) == 0 for p in all_paths),
    "verdict_line_first_pct": 100.0 * sum(bool(_VERDICT_LINE_RE.search(x)) for x in first_lines) / max(1, len(first_lines)),
    "tie_rate_pct": {key: tie_rate(key) for key in [k for k, _, _ in STRATEGY_KEYS] + ["prm_product_logprob"]},
    "seconds": {"model_load": LOAD_SECONDS, "run_loop_this_session": RUN_SECONDS,
                "sum_of_problem_wall": total(["wall_seconds"])},
    "warnings": RUN_WARNINGS,
}
with open(f"{OUT_DIR}/report_stats.json", "w") as handle:
    json.dump(report_stats, handle, indent=2)

print(json.dumps(results_summary, indent=1))
print(json.dumps(aggregation_ablation, indent=1))
print("curve:", [(c["N"], c["Accuracy (%)"]) for c in sc_scaling_curve])
print(json.dumps({k: bonus[k] for k in ("Accuracy (%)", "Fallback Steps", "Probability Agrees With Verdict (%)")}))
print(json.dumps(differences, indent=1))

In [ ]:
# [cell: export]
if os.path.isdir("/kaggle/working"):
    target = "/kaggle/working/outputs"
    if os.path.isdir(target):
        shutil.rmtree(target)
    shutil.copytree(OUT_DIR, target)
    archive = shutil.make_archive(f"/kaggle/working/{ARCHIVE_NAME}", "zip", target)
    print("Archive:", archive, os.path.getsize(archive), "bytes")
print(sorted(os.listdir(OUT_DIR)))